# M07 – Exceptions and Defensive Programming

**PCEP Alignment**: Section 4.3, 4.4 (exception hierarchy, try/except, propagation)

---

## Learning Outcomes

- Recognize built-in exceptions and when they occur.
- Use try/except/finally without hiding bugs.
- Validate inputs and handle errors at boundaries.

---

## Table of Contents

1. Exception Hierarchy (PCEP 4.3)
2. Common Built-in Exceptions
3. try / except Syntax
4. Catching Multiple Exceptions and Order
5. try / except / else / finally
6. raise Statement
7. Propagation and Delegation
8. Built-ins: exception args, re-raise
9. Practice

---

## 1. Exception Hierarchy (PCEP 4.3)

- **BaseException** – root. Subclasses: **Exception**, **KeyboardInterrupt**, **SystemExit**, etc.
- **Exception** – base for most "normal" exceptions. Subclasses:
  - **ArithmeticError** (e.g. **ZeroDivisionError**)
  - **LookupError** (e.g. **IndexError**, **KeyError**)
  - **ValueError** – valid type but invalid value
  - **TypeError** – wrong type
  - **OSError** – I/O and OS errors (e.g. **FileNotFoundError**)
- Catch **specific** exceptions first; use **Exception** only at a high boundary if needed.
- **Avoid bare except:** – it catches everything including KeyboardInterrupt.

In [ ]:
# Quick reference: common exceptions
# ZeroDivisionError - division by zero
# ValueError - e.g. int('abc')
# TypeError - e.g. 3 + 'x'
# IndexError - list index out of range
# KeyError - missing dict key
# FileNotFoundError - open('missing.txt')
print("See comments above.")

**Additional Example**: try/except for ValueError.


In [ ]:
raw = 'abc'
try:
    n = int(raw)
except ValueError:
    print(f'Cannot convert {raw!r} to int')


---

## 2. Common Built-in Exceptions

| Exception | Typical cause |
|-----------|----------------|
| **ValueError** | int("abc"), float("x"), invalid value for operation |
| **TypeError** | 3 + "a", wrong number of arguments |
| **IndexError** | lst[10] when len(lst) < 11 |
| **KeyError** | d["missing"] when key not in dict |
| **ZeroDivisionError** | 1 / 0, 10 % 0 |
| **FileNotFoundError** | open("missing.txt") when file does not exist |
| **AttributeError** | obj.no_such_attr |
| **NameError** | reference to undefined variable |

Use **getattr(obj, 'name', default)** or **.get()** for dicts to avoid AttributeError/KeyError when appropriate.

In [ ]:
try:
    int("not a number")
except ValueError as e:
    print("Caught ValueError:", e)

try:
    [1, 2][10]
except IndexError as e:
    print("Caught IndexError:", e)

**Additional Example**: except specific type first.


In [ ]:
try:
    items = []
    print(items[0])
except IndexError:
    print('List index out of range')
except Exception:
    print('Other error')


---

## 3. try / except Syntax

```python
try:
    risky_code()
except SomeError as e:
    handle(e)
```

- **as e** – bind the exception instance to e (optional but useful for logging).
- If **SomeError** (or a subclass) is raised in try, the except block runs and execution continues after the try/except.
- If a different exception is raised, it **propagates** (is not caught).

In [ ]:
def safe_divide(a: float, b: float):
    try:
        return a / b
    except ZeroDivisionError as e:
        print("Error:", e)
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

**Additional Example**: else runs when no exception.


In [ ]:
try:
    n = int('42')
except ValueError:
    print('bad input')
else:
    print('Parsed successfully:', n)


---

## 4. Catching Multiple Exceptions and Order

- **except (A, B):** – catch A or B.
- **Order matters**: put more **specific** exceptions before more **general** ones. If you put **except Exception** first, it would catch everything and later except blocks would never run.
- Example order: except ValueError: ... except TypeError: ... except Exception: ...

In [ ]:
def parse(s):
    try:
        return int(s)
    except ValueError:
        try:
            return float(s)
        except ValueError:
            return None

print(parse("42"), parse("3.14"), parse("abc"))

**Additional Example**: finally always runs.


In [ ]:
try:
    f = open('nonexistent_file_xyz.txt')
except FileNotFoundError:
    print('File missing')
finally:
    print('Cleanup block executed')


---

## 5. try / except / else / finally

- **else** – runs only if **no** exception was raised in try. Use for code that must run on success.
- **finally** – runs **always** (after try/except/else, even on return or exception). Use for cleanup (e.g. close file).

Order: try -> except -> else -> finally.

In [ ]:
try:
    x = 1 / 2
except ZeroDivisionError:
    print("Division by zero")
else:
    print("Success, x =", x)
finally:
    print("Finally always runs")

**Additional Example**: raise with message.


In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError('age must be non-negative')
    return age

try:
    set_age(-1)
except ValueError as e:
    print(e)


---

## 6. raise Statement

- **raise SomeError("message")** – raise an exception. Can be caught by caller.
- **raise** (no argument) – re-raise the current exception (inside an except block).
- Prefer **validation** before operations: if invalid, raise **ValueError** with a clear message.

In [ ]:
def divide(a: float, b: float) -> float:
    if b == 0:
        raise ValueError("b must not be zero")
    return a / b

try:
    divide(1, 0)
except ValueError as e:
    print("Caught:", e)

**Additional Example**: Defensive input validation helper.


In [ ]:
def parse_positive_int(text):
    try:
        n = int(text)
    except ValueError:
        return None
    return n if n > 0 else None

print(parse_positive_int('10'), parse_positive_int('-3'), parse_positive_int('x'))


---

## 7. Propagation and Delegation

- Unhandled exceptions **propagate** up the call stack until caught or the program exits.
- **Delegation**: let low-level code raise; catch at a **boundary** (e.g. main loop, API handler) where you can log and respond to the user.
- Do not use **except: pass** – it hides all errors. At least log or re-raise.

In [ ]:
def read_file(path: str) -> str:
    with open(path) as f:
        return f.read()  # OSError propagates if file missing

# Caller decides:
# try:
#     content = read_file("missing.txt")
# except OSError as e:
#     print("File error:", e)

---

## 8. Practice

1. Write a function that returns the first element of a list or None if the list is empty (without letting IndexError propagate).
2. Write a loop that uses try/except to read an integer from input (simulate with a list of strings); keep prompting until valid.
3. Use try/except/finally to ensure a message is printed whether or not an exception occurs.
4. Raise ValueError with a message when a function argument is out of valid range.

In [ ]:
# Practice 1: first element or None
def first(lst):
    if not lst:
        return None
    return lst[0]

print(first([1, 2, 3]), first([]))

In [ ]:
# Practice 2: read int with retry (simulated)
inputs = ["abc", "12"]
for raw in inputs:
    try:
        num = int(raw)
        print("Got:", num)
        break
    except ValueError:
        print("Invalid, try again.")

In [ ]:
# Practice 3: try/except/finally
try:
    x = 1 / 0
except ZeroDivisionError:
    print("Error caught")
finally:
    print("Cleanup done")

In [ ]:
# Practice 4: raise ValueError
def set_score(score: int) -> None:
    if not 0 <= score <= 100:
        raise ValueError("score must be 0-100")
    print("Score set:", score)

try:
    set_score(105)
except ValueError as e:
    print(e)

**Additional Example**: Exception as part of API contract.


In [ ]:
def divide(a, b):
    if b == 0:
        raise ZeroDivisionError('divisor is zero')
    return a / b

print(divide(10, 2))
try:
    divide(10, 0)
except ZeroDivisionError as e:
    print('Caught:', e)


---

## More Examples

**Example: Catching (ValueError, TypeError) together**

In [ ]:
def safe_int(s):
    try:
        return int(s)
    except (ValueError, TypeError):
        return None

print(safe_int("42"))
print(safe_int("abc"))
print(safe_int(None))

**Example: try/except/else – process only when conversion succeeds**

In [ ]:
def process_number(s):
    try:
        n = int(s)
    except ValueError:
        print("Invalid number")
    else:
        print("Double:", n * 2)

process_number("10")
process_number("ten")

---

## More Practice

**Practice 5:** Write a function that returns lst[i] if 0 <= i < len(lst), else None. Do not use try/except (use condition).

In [ ]:
def get_at(lst, i):
    if 0 <= i < len(lst):
        return lst[i]
    return None

print(get_at([10, 20, 30], 1))
print(get_at([10, 20, 30], 10))

**Practice 6:** Use try/except to convert a string to float; if it fails, print "Invalid" and use 0.0 as default.

In [ ]:
def to_float(s, default=0.0):
    try:
        return float(s)
    except (ValueError, TypeError):
        print("Invalid")
        return default

print(to_float("3.14"))
print(to_float("x"))

**Practice 7:** Raise ValueError with message "x must be positive" when x <= 0 in a function that returns 1/x.

In [ ]:
def inv(x):
    if x <= 0:
        raise ValueError("x must be positive")
    return 1 / x

print(inv(2))
try:
    inv(0)
except ValueError as e:
    print("Caught:", e)